# 🌲 Scripts — Florestas Públicas do Brasil (CNFP 2024)

**Fonte:** Serviço Florestal Brasileiro / MMA · CNFP 2024 · EPSG:4674  
**Saída:** GeoJSONs simplificados + CSVs de cache + WebMap HTML para GitHub Pages  

Execute as células em ordem. O bloco mais lento é a **Célula 4** (simplificação + dissolve, ~5–15 min).


In [ ]:
# =============================================================================
#  CÉLULA 1 — IMPORTAÇÕES, CAMINHOS E CONFIGURAÇÃO
# =============================================================================

import json, warnings, re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.validation import make_valid

warnings.filterwarnings('ignore')

BASE_GEO = Path(r'.../data')
BASE_OUT = Path(r'.../outputs')
BASE_DOCS = Path(r'.../docs')
BASE_DOCS.mkdir(parents=True, exist_ok=True)

SHP_CNFP = BASE_GEO / 'cnfp_2024' / 'cnfp_2024.shp'
SHP_MUN  = BASE_GEO / 'BR_Municipios_2024' / 'BR_Municipios_2024.shp'
SHP_UF   = BASE_GEO / 'BR_UF_2024' / 'BR_UF_2024.shp'

CRS_PROJ  = 'EPSG:5641'
CRS_GEO   = 'EPSG:4674'
BR_AREA_HA = 851_576_700  # área oficial do Brasil em ha

# Paleta por legenda (6 categorias)
LEGENDA_COLORS = {
    'Assentamento':            '#f59e0b',
    'Unidade de Conservação':  '#4ade80',
    'Florestas Não Destinadas':'#60a5fa',
    'Terra Indígena':          '#f87171',
    'Outra Destinação':        '#a78bfa',
    'Área Militar':            '#94a3b8',
}

# Simplificação: tolerância em graus (0.005° ≈ 550m — adequado para nacional)
# Para GitHub Pages precisamos de GeoJSON < 50MB
SIMPLIFY_TOL_UF  = 0.02   # estados — choropleth de fundo
SIMPLIFY_TOL_MUN = 0.01   # municípios
SIMPLIFY_TOL_FP  = 0.005  # florestas públicas (camada principal)

def ok(m):   print(f'  ✔  {m}')
def warn(m): print(f'  ⚠  {m}')
def info(m): print(f'  →  {m}')
def sec(t):  print(f'\n{"═"*70}\n  {t}\n{"═"*70}')
def safe(v, dec=2):
    try:
        f = float(v)
        return round(f, dec) if not (np.isnan(f) or np.isinf(f)) else None
    except Exception:
        return None

ok('Configuração carregada.')
print(f'  Output : {BASE_OUT}')
print(f'  Docs   : {BASE_DOCS}')


In [ ]:
# =============================================================================
#  CÉLULA 2 — CARREGAR, CORRIGIR GEOMETRIAS E CALCULAR ÁREA
# =============================================================================
sec('CARREGANDO E LIMPANDO CNFP 2024')

print('  Carregando shapefile (~30s)...')
gdf = gpd.read_file(SHP_CNFP)
ok(f'{len(gdf):,} feições carregadas | CRS: {gdf.crs}')

# ── Corrigir geometrias inválidas ─────────────────────────────────────────────
n_inv = (~gdf.geometry.is_valid).sum()
info(f'Corrigindo {n_inv} geometrias inválidas com make_valid()...')
gdf['geometry'] = gdf.geometry.apply(make_valid)
ok(f'Geometrias corrigidas. Inválidas restantes: {(~gdf.geometry.is_valid).sum()}')

# ── Calcular área em ha (projeção Albers IBGE) ────────────────────────────────
info('Calculando área em hectares (EPSG:5641)...')
gdf_proj = gdf.to_crs(CRS_PROJ)
gdf['area_calc_ha'] = (gdf_proj.geometry.area / 10_000).round(2)
total_ha = gdf['area_calc_ha'].sum()
ok(f'Área total: {total_ha:,.0f} ha | {total_ha/BR_AREA_HA*100:.2f}% do Brasil')

# ── Limpar campo estagio (tem entradas livres inválidas) ──────────────────────
ESTAGIO_VALIDOS = {
    'CERTIFICADA', 'REGULARIZADA', 'ARRECADADA', 'ESTUDO DE DELIMITAÇÃO',
    'DECLARADA', 'IDENTIFICACAO', 'DELIMITADA', 'DEMARCACAO', 'REGISTRADA',
    'ENCAMINHADA RI', 'HOMOLOGADA', 'TITULACAO', 'EM ESTUDO', 'ESTADO',
    'APROVACAO_FISCAL',
}
gdf['estagio_clean'] = gdf['estagio'].apply(
    lambda x: x.strip().upper() if isinstance(x, str) and x.strip().upper()
              in ESTAGIO_VALIDOS else None
)
ok(f'Campo estagio limpo: {gdf["estagio_clean"].notna().sum():,} válidos '
   f'de {len(gdf):,}')

# ── Normalizar anocriacao → ano inteiro ───────────────────────────────────────
def extract_year(val):
    if pd.isna(val): return None
    m = re.search(r'(\d{4})', str(val))
    return int(m.group(1)) if m else None

gdf['ano_criacao'] = gdf['anocriacao'].apply(extract_year)
anos_validos = gdf['ano_criacao'].dropna()
ok(f'Anos de criação: {int(anos_validos.min())} – {int(anos_validos.max())} '
   f'({gdf["ano_criacao"].notna().sum():,} registros)')

# ── Geocodigo como string de 7 dígitos (padrão IBGE) ─────────────────────────
gdf['cd_mun'] = gdf['geocodigo'].astype('Int64').astype(str).str.zfill(7)
ok('Campo cd_mun criado para join com municípios.')

print(f'\n  Colunas finais: {list(gdf.columns)}')


In [ ]:
# =============================================================================
#  CÉLULA 3 — ESTATÍSTICAS AGREGADAS (CSVs de cache)
# =============================================================================
sec('ESTATÍSTICAS AGREGADAS')

# ── 3a. Por UF ────────────────────────────────────────────────────────────────
info('Agregando por UF...')
df_uf = (gdf.groupby('uf')
         .agg(
             n_feicoes   =('area_calc_ha', 'count'),
             area_ha     =('area_calc_ha', 'sum'),
             area_max_ha =('area_calc_ha', 'max'),
             area_med_ha =('area_calc_ha', 'mean'),
         )
         .reset_index()
         .sort_values('area_ha', ascending=False))
df_uf['pct_total'] = (df_uf['area_ha'] / total_ha * 100).round(3)

# Área da UF para calcular % do território estadual coberto
gdf_uf = gpd.read_file(SHP_UF).to_crs(CRS_PROJ)
gdf_uf['area_uf_ha'] = gdf_uf.geometry.area / 10_000
uf_areas = gdf_uf.set_index('SIGLA_UF')['area_uf_ha'].to_dict()
df_uf['area_uf_ha']    = df_uf['uf'].map(uf_areas)
df_uf['pct_uf_coberta'] = (df_uf['area_ha'] / df_uf['area_uf_ha'] * 100).round(2)
df_uf.to_csv(BASE_OUT / 'stats_uf.csv', index=False)
ok(f'stats_uf.csv: {len(df_uf)} estados')

# ── 3b. Por UF × Legenda ──────────────────────────────────────────────────────
info('Agregando por UF × Legenda...')
df_uf_leg = (gdf.groupby(['uf', 'legenda'])['area_calc_ha']
             .sum().unstack(fill_value=0).reset_index())
df_uf_leg.columns.name = None
df_uf_leg.to_csv(BASE_OUT / 'stats_uf_legenda.csv', index=False)
ok(f'stats_uf_legenda.csv: {df_uf_leg.shape}')

# ── 3c. Por Bioma × Legenda ───────────────────────────────────────────────────
info('Agregando por Bioma × Legenda...')
df_bioma = (gdf.groupby(['bioma', 'legenda'])['area_calc_ha']
            .sum().unstack(fill_value=0).reset_index())
df_bioma.columns.name = None
df_bioma.to_csv(BASE_OUT / 'stats_bioma.csv', index=False)
ok(f'stats_bioma.csv: {df_bioma.shape}')

# ── 3d. Por Município (via geocodigo) ─────────────────────────────────────────
info('Agregando por Município...')
df_mun_raw = (gdf.groupby(['cd_mun', 'municipio', 'uf'])
              .agg(
                  n_feicoes=('area_calc_ha', 'count'),
                  area_ha  =('area_calc_ha', 'sum'),
              )
              .reset_index()
              .sort_values('area_ha', ascending=False))

# Carregar área oficial dos municípios (do shapefile IBGE)
gdf_mun = gpd.read_file(SHP_MUN).to_crs(CRS_PROJ)
gdf_mun['area_mun_ha'] = gdf_mun.geometry.area / 10_000
mun_areas = gdf_mun.set_index('CD_MUN')['area_mun_ha'].to_dict()
df_mun_raw['area_mun_ha']    = df_mun_raw['cd_mun'].map(mun_areas)
df_mun_raw['pct_mun_coberta'] = (
    df_mun_raw['area_ha'] / df_mun_raw['area_mun_ha'] * 100
).clip(0, 100).round(2)
df_mun_raw.to_csv(BASE_OUT / 'stats_municipios.csv', index=False)
ok(f'stats_municipios.csv: {len(df_mun_raw):,} municípios com floresta pública')

# ── 3e. Série histórica de criação ───────────────────────────────────────────
info('Série histórica de criação...')
df_serie = (gdf[gdf['ano_criacao'].notna()]
            .groupby('ano_criacao')
            .agg(n_criadas=('area_calc_ha','count'),
                 area_ha  =('area_calc_ha','sum'))
            .reset_index()
            .rename(columns={'ano_criacao':'ano'})
            .sort_values('ano'))
df_serie['area_acum_ha'] = df_serie['area_ha'].cumsum()
df_serie.to_csv(BASE_OUT / 'stats_serie_historica.csv', index=False)
ok(f'stats_serie_historica.csv: {len(df_serie)} anos | '
   f'{int(df_serie["ano"].min())}–{int(df_serie["ano"].max())}')

# ── 3f. KPIs nacionais ────────────────────────────────────────────────────────
kpis = {
    'total_ha':          round(total_ha, 0),
    'total_km2':         round(total_ha / 100, 0),
    'pct_brasil':        round(total_ha / BR_AREA_HA * 100, 2),
    'n_feicoes':         len(gdf),
    'n_municipios':      df_mun_raw['cd_mun'].nunique(),
    'n_ufs':             gdf['uf'].nunique(),
    'area_media_ha':     round(gdf['area_calc_ha'].mean(), 1),
    'maior_ha':          round(gdf['area_calc_ha'].max(), 0),
    'maior_nome':        gdf.loc[gdf['area_calc_ha'].idxmax(), 'nome'],
    'maior_uf':          gdf.loc[gdf['area_calc_ha'].idxmax(), 'uf'],
    'data_processamento': datetime.now().strftime('%d/%m/%Y'),
}
pd.DataFrame([kpis]).to_csv(BASE_OUT / 'kpis_nacionais.csv', index=False)

print(f'\n  ┌─ KPIs Nacionais')
for k, v in kpis.items():
    print(f'     {k:<25}: {v}')


In [ ]:
# =============================================================================
#  CÉLULA 4 — SIMPLIFICAÇÃO E EXPORT GEOJSON (GitHub Pages)
#  Estratégia: 3 camadas separadas por tamanho de tolerância
#  Meta: todos os GeoJSONs < 30MB cada
# =============================================================================
sec('SIMPLIFICAÇÃO E EXPORT GEOJSON')

# ── 4a. GeoJSON de UFs (background choropleth) ────────────────────────────────
info('GeoJSON de estados...')
gdf_uf_web = gpd.read_file(SHP_UF).to_crs(CRS_GEO)
gdf_uf_web['geometry'] = gdf_uf_web.geometry.simplify(SIMPLIFY_TOL_UF,
                                                         preserve_topology=True)
gdf_uf_web = gdf_uf_web.merge(
    df_uf[['uf','n_feicoes','area_ha','pct_total','pct_uf_coberta']]
      .rename(columns={'uf':'SIGLA_UF'}),
    on='SIGLA_UF', how='left'
)
path_uf = BASE_OUT / 'geojson_uf.json'
gdf_uf_web[['SIGLA_UF','NM_UF','NM_REGIA','n_feicoes',
             'area_ha','pct_total','pct_uf_coberta','geometry']]\
    .to_file(path_uf, driver='GeoJSON')
sz = path_uf.stat().st_size / (1024**2)
ok(f'geojson_uf.json: {sz:.1f} MB')

# ── 4b. GeoJSON de municípios (para choropleth municipal) ─────────────────────
info('GeoJSON de municípios (simplificado)...')
gdf_mun_web = gpd.read_file(SHP_MUN).to_crs(CRS_GEO)
gdf_mun_web['geometry'] = gdf_mun_web.geometry.simplify(SIMPLIFY_TOL_MUN,
                                                           preserve_topology=True)
gdf_mun_web = gdf_mun_web.merge(
    df_mun_raw[['cd_mun','area_ha','n_feicoes','pct_mun_coberta']],
    left_on='CD_MUN', right_on='cd_mun', how='left'
)
gdf_mun_web['area_ha']       = gdf_mun_web['area_ha'].fillna(0)
gdf_mun_web['pct_mun_coberta'] = gdf_mun_web['pct_mun_coberta'].fillna(0)
path_mun = BASE_OUT / 'geojson_municipios.json'
gdf_mun_web[['CD_MUN','NM_MUN','SIGLA_UF','area_ha',
              'n_feicoes','pct_mun_coberta','geometry']]\
    .to_file(path_mun, driver='GeoJSON')
sz = path_mun.stat().st_size / (1024**2)
ok(f'geojson_municipios.json: {sz:.1f} MB')
if sz > 40:
    warn(f'Arquivo grande ({sz:.1f}MB). Aumentando tolerância de simplificação...')
    gdf_mun_web['geometry'] = gdf_mun_web.geometry.simplify(0.02,
                                                              preserve_topology=True)
    gdf_mun_web[['CD_MUN','NM_MUN','SIGLA_UF','area_ha',
                  'n_feicoes','pct_mun_coberta','geometry']]\
        .to_file(path_mun, driver='GeoJSON')
    sz2 = path_mun.stat().st_size / (1024**2)
    ok(f'Re-simplificado: {sz2:.1f} MB')

# ── 4c. GeoJSON das florestas públicas (camada principal) ─────────────────────
info('GeoJSON CNFP simplificado (camada principal)...')
cols_export = ['nome','legenda','protecao','governo','bioma','uf',
               'municipio','area_calc_ha','comunitari','classe',
               'ano_criacao','estagio_clean','sobreposic','codigo','geometry']
gdf_web = gdf[cols_export].copy()
gdf_web['geometry'] = gdf_web.geometry.simplify(SIMPLIFY_TOL_FP,
                                                   preserve_topology=True)
# Remover polígonos que viraram pontos/linhas após simplificação
gdf_web = gdf_web[gdf_web.geometry.geom_type.isin(
    ['Polygon','MultiPolygon'])].reset_index(drop=True)

path_fp = BASE_OUT / 'geojson_cnfp.json'
gdf_web.to_file(path_fp, driver='GeoJSON')
sz = path_fp.stat().st_size / (1024**2)
ok(f'geojson_cnfp.json: {sz:.1f} MB | {len(gdf_web):,} feições')

if sz > 80:
    warn(f'GeoJSON muito grande ({sz:.1f}MB) para GitHub Pages. '
         f'Re-simplificando com tolerância maior...')
    gdf_web['geometry'] = gdf_web.geometry.simplify(0.01, preserve_topology=True)
    gdf_web = gdf_web[gdf_web.geometry.geom_type.isin(
        ['Polygon','MultiPolygon'])].reset_index(drop=True)
    gdf_web.to_file(path_fp, driver='GeoJSON')
    sz2 = path_fp.stat().st_size / (1024**2)
    ok(f'Re-simplificado: {sz2:.1f} MB')

print(f'\n  Arquivos GeoJSON gerados em: {BASE_OUT}')


In [ ]:
# =============================================================================
#  CÉLULA 5 — PREPARAR DADOS JAVASCRIPT
# =============================================================================
sec('PREPARANDO DADOS PARA O WEBMAP')

# ── Recarregar CSVs ───────────────────────────────────────────────────────────
df_uf_js     = pd.read_csv(BASE_OUT / 'stats_uf.csv')
df_bioma_js  = pd.read_csv(BASE_OUT / 'stats_bioma.csv')
df_mun_js    = pd.read_csv(BASE_OUT / 'stats_municipios.csv')
df_serie_js  = pd.read_csv(BASE_OUT / 'stats_serie_historica.csv')
df_uf_leg_js = pd.read_csv(BASE_OUT / 'stats_uf_legenda.csv')
kpis_js      = pd.read_csv(BASE_OUT / 'kpis_nacionais.csv').iloc[0].to_dict()

# ── Stats por UF para JS ──────────────────────────────────────────────────────
stats_uf = {}
for _, r in df_uf_js.iterrows():
    stats_uf[r['uf']] = {
        'n':       int(r['n_feicoes']),
        'area':    round(float(r['area_ha']), 0),
        'pct_br':  round(float(r['pct_total']), 3),
        'pct_uf':  round(float(r['pct_uf_coberta']), 2),
    }

# Breakdown por legenda por UF
leg_cols = [c for c in df_uf_leg_js.columns if c != 'uf']
for _, r in df_uf_leg_js.iterrows():
    uf = r['uf']
    if uf in stats_uf:
        stats_uf[uf]['leg'] = {c: round(float(r[c]), 0) for c in leg_cols}

# ── Ranking municípios (top 200 por área) ─────────────────────────────────────
rank_mun_area = (
    df_mun_js.nlargest(200, 'area_ha')
    [['cd_mun','municipio','uf','area_ha','pct_mun_coberta','n_feicoes']]
    .to_dict(orient='records')
)
rank_mun_pct = (
    df_mun_js[df_mun_js['pct_mun_coberta'] > 0]
    .nlargest(200, 'pct_mun_coberta')
    [['cd_mun','municipio','uf','area_ha','pct_mun_coberta','n_feicoes']]
    .to_dict(orient='records')
)

# ── Série histórica ──────────────────────────────────────────────────────────
serie_hist = df_serie_js[['ano','n_criadas','area_ha','area_acum_ha']]\
    .astype({'ano': int}).to_dict(orient='records')

# ── Bioma breakdown ───────────────────────────────────────────────────────────
bioma_data = {}
for _, r in df_bioma_js.iterrows():
    bioma_data[r['bioma']] = {c: round(float(r[c]), 0)
                               for c in df_bioma_js.columns if c != 'bioma'}

ok('Dados preparados para JavaScript.')
print(f'  UFs       : {len(stats_uf)}')
print(f'  Municípios: {len(rank_mun_area)} (top 200 por área)')
print(f'  Série     : {len(serie_hist)} anos')
print(f'  Biomas    : {len(bioma_data)}')


In [ ]:
# =============================================================================
#  CÉLULA 6 — GERAR WEBMAP HTML (GitHub Pages compatible)
# =============================================================================
sec('GERANDO WEBMAP HTML')

# Serializar para JS
STATS_UF_JS      = json.dumps(stats_uf,       ensure_ascii=False)
RANK_AREA_JS     = json.dumps(rank_mun_area,   ensure_ascii=False)
RANK_PCT_JS      = json.dumps(rank_mun_pct,    ensure_ascii=False)
SERIE_JS         = json.dumps(serie_hist,      ensure_ascii=False)
BIOMA_JS         = json.dumps(bioma_data,      ensure_ascii=False)
LEG_COLORS_JS    = json.dumps(LEGENDA_COLORS,  ensure_ascii=False)

total_ha_fmt = f"{kpis_js['total_ha']/1e6:.2f}M"
pct_br       = f"{kpis_js['pct_brasil']:.2f}"
n_mun        = f"{int(kpis_js['n_municipios']):,}".replace(',', '.')
n_feat       = f"{int(kpis_js['n_feicoes']):,}".replace(',', '.')
data_hoje    = kpis_js['data_processamento']

# Nota: o webmap carrega os GeoJSONs externos via fetch()
# Para GitHub Pages, os arquivos GeoJSON ficam na mesma pasta do HTML
# ou em subpasta 'data/' — ajuste os paths abaixo conforme a estrutura do repo

HTML = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Florestas Públicas do Brasil · CNFP 2024</title>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-annotation@3.0.1/dist/chartjs-plugin-annotation.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Source+Sans+3:wght@300;400;600;700&family=Source+Code+Pro:wght@400;600&display=swap" rel="stylesheet">

<style>
*,*::before,*::after{{margin:0;padding:0;box-sizing:border-box;}}
:root{{
  --bg:      #050d08;
  --bg2:     #0a1a0d;
  --bg3:     #0f2214;
  --border:  #163520;
  --border2: #1f4a2c;
  --green:   #4ade80;
  --green2:  #22c55e;
  --forest:  #166534;
  --teal:    #2dd4bf;
  --amber:   #f59e0b;
  --blue:    #60a5fa;
  --red:     #f87171;
  --purple:  #a78bfa;
  --slate:   #94a3b8;
  --txt:     #dcfce7;
  --txt2:    #86efac;
  --muted:   #3d6b4a;
  --font:    'Source Sans 3', sans-serif;
  --mono:    'Source Code Pro', monospace;
}}
body{{font-family:var(--font);background:var(--bg);color:var(--txt);
      height:100vh;display:flex;flex-direction:column;overflow:hidden;}}

/* ── HEADER ── */
#hdr{{
  height:54px;flex-shrink:0;z-index:600;
  background:linear-gradient(90deg,#020805 0%,#081409 50%,#050d08 100%);
  border-bottom:1px solid var(--border);
  display:flex;align-items:center;gap:16px;padding:0 20px;
  box-shadow:0 2px 20px rgba(0,0,0,.6);
}}
.hdr-icon{{font-size:24px;}}
.hdr-t h1{{font-family:var(--font);font-size:15px;font-weight:700;
           color:#bbf7d0;letter-spacing:-.1px;line-height:1.2;}}
.hdr-t p{{font-size:9.5px;color:var(--muted);margin-top:1px;}}
.hdr-kpis{{margin-left:auto;display:flex;gap:22px;align-items:center;}}
.hkpi{{text-align:center;}}
.hkpi-v{{font-family:var(--mono);font-size:17px;font-weight:600;
          color:var(--green);line-height:1;}}
.hkpi-v.amber{{color:var(--amber);}}
.hkpi-v.blue{{color:var(--blue);}}
.hkpi-l{{font-size:9px;color:var(--muted);margin-top:2px;}}

/* ── LAYOUT ── */
#main{{display:flex;flex:1;overflow:hidden;}}
#map {{flex:1;position:relative;}}
#panel{{
  width:420px;background:var(--bg2);border-left:1px solid var(--border);
  display:flex;flex-direction:column;overflow:hidden;
  box-shadow:-4px 0 24px rgba(0,0,0,.5);
}}

/* ── TABS ── */
#tabs{{display:flex;background:var(--bg);border-bottom:1px solid var(--border);flex-shrink:0;}}
.tab{{flex:1;padding:10px 4px;text-align:center;font-size:10px;font-weight:600;
      letter-spacing:.5px;text-transform:uppercase;color:var(--muted);
      cursor:pointer;border-bottom:2px solid transparent;transition:all .2s;}}
.tab.active{{color:var(--green);border-bottom-color:var(--green);background:rgba(74,222,128,.04);}}
.tab:hover:not(.active){{color:var(--txt2);}}

/* ── CONTENT ── */
.tc{{flex:1;overflow-y:auto;display:none;flex-direction:column;gap:10px;padding:13px;}}
.tc.active{{display:flex;}}
.tc::-webkit-scrollbar{{width:3px;}}
.tc::-webkit-scrollbar-thumb{{background:var(--border2);border-radius:2px;}}

/* ── CARD ── */
.card{{background:var(--bg3);border:1px solid var(--border);border-radius:10px;
       padding:13px;flex-shrink:0;}}
.card h4{{font-size:9.5px;font-weight:700;letter-spacing:.7px;text-transform:uppercase;
          color:var(--muted);margin-bottom:10px;}}

/* ── MODE BTNS ── */
.mbtn{{flex:1;padding:7px;border-radius:7px;border:1px solid var(--border2);
       background:var(--bg);color:var(--muted);font-family:var(--font);
       font-size:11px;font-weight:600;cursor:pointer;transition:all .2s;}}
.mbtn.on{{background:var(--forest);border-color:var(--green2);color:var(--txt);}}

/* ── KPI GRID ── */
.kgrid{{display:grid;grid-template-columns:1fr 1fr;gap:7px;}}
.kpi{{background:var(--bg);border:1px solid var(--border);border-radius:8px;
      padding:10px;text-align:center;}}
.kval{{font-family:var(--mono);font-size:19px;font-weight:600;color:var(--green);line-height:1;}}
.kval.amber{{color:var(--amber);}} .kval.blue{{color:var(--blue);}}
.kval.purple{{color:var(--purple);}} .kval.teal{{color:var(--teal);}}
.klbl{{font-size:9px;color:var(--muted);margin-top:3px;}}

/* ── NARR ── */
.narr{{padding:10px 12px;border-left:2px solid var(--forest);border-radius:0 8px 8px 0;
       background:rgba(22,101,52,.1);font-size:11.5px;line-height:1.65;color:#a7f3d0;}}

/* ── RANK ── */
.ritem{{display:flex;align-items:center;gap:7px;padding:7px 8px;border-radius:7px;
        margin-bottom:2px;background:var(--bg);border:1px solid var(--border);
        cursor:pointer;transition:border-color .15s;}}
.ritem:hover{{border-color:var(--green2);}}
.rnum{{font-family:var(--mono);font-size:10px;color:var(--muted);width:18px;text-align:right;}}
.rname{{flex:1;font-size:11.5px;color:var(--txt);}}
.rname small{{display:block;font-size:9px;color:var(--muted);}}
.rval{{font-family:var(--mono);font-size:13px;font-weight:600;color:var(--green);white-space:nowrap;}}
.rbar{{height:2px;background:var(--border);border-radius:1px;margin-top:3px;}}
.rfill{{height:2px;background:var(--forest);border-radius:1px;transition:width .5s;}}

/* ── STAT ROW ── */
.srow{{display:flex;justify-content:space-between;align-items:center;
       padding:5px 0;border-bottom:1px solid var(--border);font-size:11px;}}
.srow:last-child{{border-bottom:none;}}
.slbl{{color:var(--muted);}} .sval{{font-weight:600;color:#bbf7d0;}}

/* ── CHART ── */
.cwrap{{position:relative;}}
.h180{{height:180px;}} .h200{{height:200px;}} .h240{{height:240px;}}

/* ── LEGEND ── */
#leg{{
  position:absolute;bottom:28px;right:428px;z-index:800;
  background:rgba(5,13,8,.94);border:1px solid var(--border2);
  border-radius:10px;padding:12px 14px;min-width:195px;
  backdrop-filter:blur(8px);box-shadow:0 4px 16px rgba(0,0,0,.5);
}}
#leg h5{{font-size:9px;font-weight:700;letter-spacing:.7px;text-transform:uppercase;
         color:var(--muted);margin-bottom:8px;}}
.leg-item{{display:flex;align-items:center;gap:7px;margin-bottom:4px;font-size:10px;color:var(--txt2);}}
.leg-dot{{width:11px;height:11px;border-radius:3px;flex-shrink:0;}}

/* ── FILTER BAR ── */
#filter-bar{{
  position:absolute;top:10px;left:50%;transform:translateX(-50%);
  z-index:800;display:flex;gap:6px;background:rgba(5,13,8,.92);
  border:1px solid var(--border2);border-radius:20px;padding:6px 12px;
  backdrop-filter:blur(8px);box-shadow:0 2px 12px rgba(0,0,0,.4);
}}
.fbtn{{padding:4px 12px;border-radius:12px;border:1px solid transparent;
       background:transparent;color:var(--muted);font-family:var(--font);
       font-size:10px;font-weight:600;cursor:pointer;transition:all .2s;white-space:nowrap;}}
.fbtn.active{{background:var(--forest);border-color:var(--green2);color:var(--txt);}}
.fbtn:hover:not(.active){{color:var(--txt2);}}

/* ── LOADING ── */
#loading{{position:fixed;inset:0;background:var(--bg);display:flex;
          align-items:center;justify-content:center;z-index:9999;
          flex-direction:column;gap:14px;}}
.spin{{width:42px;height:42px;border:2px solid var(--border);
       border-top-color:var(--green);border-radius:50%;animation:sp .7s linear infinite;}}
@keyframes sp{{to{{transform:rotate(360deg)}}}}
#loading p{{font-size:12px;color:var(--muted);}}
#load-detail{{font-size:10px;color:var(--border2);margin-top:4px;}}

/* ── TOOLTIP ── */
.lf-tip{{
  background:#030a05 !important;border:1px solid var(--border2) !important;
  color:var(--txt) !important;font-family:var(--font) !important;
  font-size:12px !important;padding:8px 12px !important;
  border-radius:8px !important;box-shadow:0 3px 12px rgba(0,0,0,.5) !important;
}}

/* ── LEG PILLS ── */
.leg-pill{{display:inline-flex;align-items:center;gap:5px;padding:3px 8px;
           border-radius:12px;font-size:10px;margin:2px;cursor:pointer;
           border:1px solid rgba(255,255,255,.1);transition:all .2s;}}
.leg-pill.off{{opacity:.35;}}
.leg-pill:hover{{border-color:rgba(255,255,255,.3);}}
.pill-dot{{width:8px;height:8px;border-radius:50%;flex-shrink:0;}}
</style>
</head>
<body>

<div id="loading">
  <div class="spin"></div>
  <p>Carregando Florestas Públicas do Brasil...</p>
  <div id="load-detail">Aguarde — carregando GeoJSONs</div>
</div>

<!-- HEADER -->
<div id="hdr">
  <div class="hdr-icon">🌲</div>
  <div class="hdr-t">
    <h1>Cadastro Nacional de Florestas Públicas · Brasil 2024</h1>
    <p>SFB/MMA · CNFP 2024 · Processado em {data_hoje} · Fonte: Serviço Florestal Brasileiro</p>
  </div>
  <div class="hdr-kpis">
    <div class="hkpi">
      <div class="hkpi-v">{total_ha_fmt}</div>
      <div class="hkpi-l">ha florestas públicas</div>
    </div>
    <div class="hkpi">
      <div class="hkpi-v amber">{pct_br}%</div>
      <div class="hkpi-l">do território BR</div>
    </div>
    <div class="hkpi">
      <div class="hkpi-v blue">{n_feat}</div>
      <div class="hkpi-l">feições cadastradas</div>
    </div>
    <div class="hkpi">
      <div class="hkpi-v" style="color:var(--purple)">{n_mun}</div>
      <div class="hkpi-l">municípios abrangidos</div>
    </div>
  </div>
</div>

<div id="main">
  <div id="map">
    <!-- Filter bar -->
    <div id="filter-bar">
      <button class="fbtn active" onclick="setFilter('all')">Todas</button>
      <button class="fbtn" onclick="setFilter('Assentamento')">Assentamentos</button>
      <button class="fbtn" onclick="setFilter('Unidade de Conservação')">UCs</button>
      <button class="fbtn" onclick="setFilter('Florestas Não Destinadas')">Não Destinadas</button>
      <button class="fbtn" onclick="setFilter('Terra Indígena')">TIs</button>
      <button class="fbtn" onclick="setFilter('Outra Destinação')">Outras</button>
    </div>
  </div>

  <!-- PANEL -->
  <div id="panel">
    <div id="tabs">
      <div class="tab active" onclick="switchTab('visao')">🗺 VISÃO GERAL</div>
      <div class="tab" onclick="switchTab('ranking')">🏆 RANKING</div>
      <div class="tab" onclick="switchTab('historia')">📈 HISTÓRICO</div>
      <div class="tab" onclick="switchTab('bioma')">🌿 BIOMAS</div>
    </div>

    <!-- ══ TAB VISÃO GERAL ═══════════════════════════════════════════════ -->
    <div id="tc-visao" class="tc active">

      <div class="card">
        <h4>Escala de visualização</h4>
        <div style="display:flex;gap:6px">
          <button class="mbtn on" onclick="setMapMode('uf')"   id="btn-uf">📍 Estados</button>
          <button class="mbtn"    onclick="setMapMode('mun')"  id="btn-mun">🏘 Municípios</button>
          <button class="mbtn"    onclick="setMapMode('fp')"   id="btn-fp">🌲 Florestas</button>
        </div>
      </div>

      <div class="narr" id="narr-main">
        O Cadastro Nacional de Florestas Públicas abrange
        <strong>{total_ha_fmt} ha</strong> — equivalente a <strong>{pct_br}%</strong>
        do território brasileiro. Clique em um estado ou município para
        explorar a composição por categoria.
      </div>

      <div class="card">
        <h4>KPIs — <span id="kpi-sel-lbl">Brasil</span></h4>
        <div class="kgrid" id="kpi-grid"></div>
      </div>

      <div class="card">
        <h4>Composição por Categoria</h4>
        <div class="cwrap h200"><canvas id="chart-donut"></canvas></div>
      </div>

      <div class="card" id="detail-card" style="display:none">
        <h4>Detalhe — <span id="det-nome" style="color:var(--green)">—</span></h4>
        <div id="det-stats"></div>
      </div>

    </div>

    <!-- ══ TAB RANKING ═══════════════════════════════════════════════════ -->
    <div id="tc-ranking" class="tc">

      <div class="card">
        <h4>Ordenar por</h4>
        <div style="display:flex;gap:6px">
          <button class="mbtn on" onclick="setRankMode('area')" id="rbtn-area">Área (ha)</button>
          <button class="mbtn"    onclick="setRankMode('pct')"  id="rbtn-pct">% do Município</button>
        </div>
      </div>

      <div class="card">
        <h4>Top 20 Estados — Área de Floresta Pública</h4>
        <div class="cwrap h240"><canvas id="chart-uf-bar"></canvas></div>
      </div>

      <div class="card">
        <h4>Top 50 Municípios — <span id="rank-lbl">Área total (ha)</span></h4>
        <div id="ranking-mun"></div>
      </div>

    </div>

    <!-- ══ TAB HISTÓRICO ═════════════════════════════════════════════════ -->
    <div id="tc-historia" class="tc">

      <div class="card">
        <h4>Série Histórica de Criação — Área Acumulada (ha)</h4>
        <p style="font-size:11px;color:var(--muted);margin-bottom:10px;line-height:1.55">
          Evolução do cadastro de florestas públicas ao longo do tempo,
          considerando o campo <em>anocriacao</em> (registros com data informada).
        </p>
        <div class="cwrap h200"><canvas id="chart-serie-acum"></canvas></div>
      </div>

      <div class="card">
        <h4>Criações Anuais (novas feições e área)</h4>
        <div class="cwrap h200"><canvas id="chart-serie-anual"></canvas></div>
      </div>

    </div>

    <!-- ══ TAB BIOMAS ════════════════════════════════════════════════════ -->
    <div id="tc-bioma" class="tc">

      <div class="card">
        <h4>Área por Bioma (ha)</h4>
        <div class="cwrap h200"><canvas id="chart-bioma"></canvas></div>
      </div>

      <div class="card">
        <h4>Composição por Bioma × Categoria</h4>
        <div id="bioma-detail"></div>
      </div>

    </div>
  </div>
</div>

<!-- Legenda do mapa -->
<div id="leg">
  <h5>Categoria CNFP</h5>
  <div id="leg-items"></div>
  <div style="border-top:1px solid var(--border);margin:7px 0 6px"></div>
  <h5 style="margin-bottom:5px">Choropleth (%UF coberta)</h5>
  <div style="height:7px;border-radius:4px;
    background:linear-gradient(to right,#14532d,#16a34a,#fbbf24,#dc2626);
    margin-bottom:3px"></div>
  <div style="display:flex;justify-content:space-between;font-size:9px;color:var(--muted)">
    <span>0%</span><span>100%</span>
  </div>
</div>

<script>
/* ── DADOS ─────────────────────────────────────────────────────────────── */
const STATS_UF   = {STATS_UF_JS};
const RANK_AREA  = {RANK_AREA_JS};
const RANK_PCT   = {RANK_PCT_JS};
const SERIE      = {SERIE_JS};
const BIOMA_DATA = {BIOMA_JS};
const LEG_COLORS = {LEG_COLORS_JS};
const TOTAL_HA   = {round(kpis_js['total_ha'], 0)};
const PCT_BR     = {kpis_js['pct_brasil']};

/* ── ESTADO ──────────────────────────────────────────────────────────────── */
let map, layerUF, layerMUN, layerFP;
let mapMode    = 'uf';
let filterCat  = 'all';
let selectedId = null;
let rankMode   = 'area';
let cDonut=null, cUFBar=null, cSerieAcum=null, cSerieAnual=null, cBioma=null;

/* ── CORES ───────────────────────────────────────────────────────────────── */
function pctColor(pct) {{
  const t = Math.max(0, Math.min(1, pct / 100));
  const stops = [
    [0,[20,83,45]], [0.3,[22,163,74]], [0.6,[251,191,36]], [1,[220,38,38]]
  ];
  let lo=stops[0], hi=stops[stops.length-1];
  for(let i=0;i<stops.length-1;i++)
    if(t>=stops[i][0]&&t<=stops[i+1][0]){{lo=stops[i];hi=stops[i+1];break;}}
  const f=lo[0]===hi[0]?0:(t-lo[0])/(hi[0]-lo[0]);
  const r=v=>Math.round(lo[1][v]+(hi[1][v]-lo[1][v])*f);
  return `rgb(${{r(0)}},${{r(1)}},${{r(2)}})`;
}}

/* ── MAPA ────────────────────────────────────────────────────────────────── */
function initMap() {{
  map = L.map('map',{{center:[-14,-52],zoom:4,zoomControl:true,preferCanvas:false}});
  L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}{{r}}.png',
    {{attribution:'© CartoDB | SFB/MMA',opacity:0.7}}).addTo(map);

  // Legenda pills
  const legBox = document.getElementById('leg-items');
  Object.entries(LEG_COLORS).forEach(([cat,col]) => {{
    const d = document.createElement('div');
    d.className = 'leg-item';
    d.innerHTML = `<div class="leg-dot" style="background:${{col}}"></div>${{cat}}`;
    legBox.appendChild(d);
  }});
}}

async function loadLayers() {{
  const setStatus = s => document.getElementById('load-detail').textContent = s;

  // Caminhos dos GeoJSON — ajuste se o repo usar subpasta 'data/'
  setStatus('Carregando estados...');
  const rUF  = await fetch('geojson_uf.json');
  const gUF  = await rUF.json();

  setStatus('Carregando municípios...');
  const rMUN = await fetch('geojson_municipios.json');
  const gMUN = await rMUN.json();

  setStatus('Carregando florestas públicas...');
  const rFP  = await fetch('geojson_cnfp.json');
  const gFP  = await rFP.json();

  // ── Layer UF ──────────────────────────────────────────────────────────────
  layerUF = L.geoJSON(gUF, {{
    style: f => ({{
      fillColor: pctColor((STATS_UF[f.properties.SIGLA_UF]||{{}}).pct_uf||0),
      fillOpacity: 0.65, color:'#163520', weight:0.8, opacity:0.8,
    }}),
    onEachFeature: (f,l) => {{
      const s  = STATS_UF[f.properties.SIGLA_UF]||{{}};
      l.on('mouseover', e => {{
        l.setStyle({{color:'#fff',weight:2,fillOpacity:0.9}});
        l.bindTooltip(`<b style="color:#bbf7d0">${{f.properties.NM_UF}}</b><br>
          Área FP: <b>${{(s.area||0).toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha</b><br>
          % UF coberta: <b>${{(s.pct_uf||0).toFixed(1)}}%</b>`,
          {{className:'lf-tip',sticky:true}}).openTooltip(e.latlng);
      }});
      l.on('mouseout',()=>{{ layerUF.resetStyle(l); l.closeTooltip(); }});
      l.on('click',()=>selectUF(f.properties.SIGLA_UF,f.properties.NM_UF));
    }}
  }}).addTo(map);

  // ── Layer MUN ─────────────────────────────────────────────────────────────
  layerMUN = L.geoJSON(gMUN, {{
    style: f => ({{
      fillColor: pctColor(f.properties.pct_mun_coberta||0),
      fillOpacity: 0.70, color:'#0f2214', weight:0.3, opacity:0.6,
    }}),
    onEachFeature: (f,l) => {{
      l.on('mouseover', e => {{
        l.setStyle({{color:'#fff',weight:1.5,fillOpacity:0.92}});
        l.bindTooltip(`<b style="color:#bbf7d0">${{f.properties.NM_MUN}}</b>
          <span style="color:var(--muted)"> · ${{f.properties.SIGLA_UF}}</span><br>
          Área FP: <b>${{(f.properties.area_ha||0).toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha</b><br>
          % do município: <b>${{(f.properties.pct_mun_coberta||0).toFixed(1)}}%</b>`,
          {{className:'lf-tip',sticky:true}}).openTooltip(e.latlng);
      }});
      l.on('mouseout',()=>{{ layerMUN.resetStyle(l); l.closeTooltip(); }});
    }}
  }});

  // ── Layer FP (florestas públicas com filtro por legenda) ──────────────────
  layerFP = L.geoJSON(gFP, {{
    style: f => ({{
      fillColor:   LEG_COLORS[f.properties.legenda]||'#888',
      fillOpacity: filterCat==='all'||f.properties.legenda===filterCat ? 0.75 : 0.08,
      color:       LEG_COLORS[f.properties.legenda]||'#888',
      weight:      0.4, opacity:0.7,
    }}),
    onEachFeature: (f,l) => {{
      l.on('mouseover', e => {{
        l.setStyle({{weight:2,fillOpacity:0.95,color:'#fff'}});
        l.bindTooltip(`<div style="min-width:180px">
          <b style="color:#bbf7d0">${{f.properties.nome||'—'}}</b><br>
          <span style="color:var(--muted)">${{f.properties.legenda||'—'}} · ${{f.properties.bioma||'—'}}</span><br>
          Área: <b>${{(f.properties.area_calc_ha||0).toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha</b><br>
          Governo: ${{f.properties.governo||'—'}} · UF: ${{f.properties.uf||'—'}}
        </div>`,{{className:'lf-tip',sticky:true}}).openTooltip(e.latlng);
      }});
      l.on('mouseout',()=>{{layerFP.resetStyle(l);l.closeTooltip();}});
    }}
  }});

  setStatus('');
  document.getElementById('loading').style.display='none';
  buildNationalKPIs();
  buildDonut(calcNationalLeg());
}}

/* ── MAPA MODES ──────────────────────────────────────────────────────────── */
function setMapMode(m) {{
  mapMode=m;
  ['uf','mun','fp'].forEach(id => {{
    document.getElementById(`btn-${{id}}`).classList.toggle('on', id===m);
  }});
  if(layerUF)  m==='uf'  ? layerUF.addTo(map)  : map.removeLayer(layerUF);
  if(layerMUN) m==='mun' ? layerMUN.addTo(map) : map.removeLayer(layerMUN);
  if(layerFP)  m==='fp'  ? layerFP.addTo(map)  : map.removeLayer(layerFP);
}}

function setFilter(cat) {{
  filterCat=cat;
  document.querySelectorAll('.fbtn').forEach(b =>
    b.classList.toggle('active', b.textContent.trim()===
      (cat==='all'?'Todas':cat==='Unidade de Conservação'?'UCs':
       cat==='Florestas Não Destinadas'?'Não Destinadas':
       cat==='Terra Indígena'?'TIs':
       cat==='Outra Destinação'?'Outras':'Assentamentos')
    )
  );
  if(layerFP) layerFP.eachLayer(l => l.setStyle({{
    fillOpacity: cat==='all'||l.feature.properties.legenda===cat ? 0.75 : 0.05,
  }}));
}}

/* ── SELEÇÃO ─────────────────────────────────────────────────────────────── */
function selectUF(sigla, nome) {{
  const s = STATS_UF[sigla]||{{}};
  document.getElementById('kpi-sel-lbl').textContent = nome||sigla;
  const leg = s.leg||{{}};
  document.getElementById('det-nome').textContent = nome||sigla;
  document.getElementById('det-stats').innerHTML = `
    <div class="srow"><span class="slbl">Área total FP</span>
      <span class="sval">${{(s.area||0).toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha</span></div>
    <div class="srow"><span class="slbl">% UF coberta</span>
      <span class="sval">${{(s.pct_uf||0).toFixed(2)}}%</span></div>
    <div class="srow"><span class="slbl">Feições cadastradas</span>
      <span class="sval">${{(s.n||0).toLocaleString('pt-BR')}}</span></div>
    ${{Object.entries(leg).map(([k,v])=>`
      <div class="srow">
        <span class="slbl" style="display:flex;align-items:center;gap:5px">
          <span style="width:8px;height:8px;border-radius:50%;display:inline-block;
                        background:${{LEG_COLORS[k]||'#888'}}"></span>
          ${{k}}</span>
        <span class="sval">${{v.toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha</span>
      </div>`).join('')}}
  `;
  document.getElementById('detail-card').style.display='block';
  buildDonut(leg);
  document.getElementById('kpi-grid').innerHTML = kpiHTML([
    [(s.area||0).toLocaleString('pt-BR',{{maximumFractionDigits:0}}),'Área FP (ha)',''],
    [(s.pct_uf||0).toFixed(1)+'%','% UF coberta','amber'],
    [(s.n||0).toLocaleString('pt-BR'),'Feições','blue'],
    [(s.pct_br||0).toFixed(2)+'%','% do total BR','teal'],
  ]);
}}

/* ── KPIs ────────────────────────────────────────────────────────────────── */
function buildNationalKPIs() {{
  document.getElementById('kpi-grid').innerHTML = kpiHTML([
    [TOTAL_HA.toLocaleString('pt-BR',{{maximumFractionDigits:0}}),'Área total (ha)',''],
    [PCT_BR.toFixed(2)+'%','% do Brasil','amber'],
    [{n_feat},'Feições','blue'],
    [{n_mun},'Municípios','purple'],
  ]);
}}
function kpiHTML(items) {{
  return items.map(([v,l,c])=>
    `<div class="kpi"><div class="kval ${{c}}">${{v}}</div><div class="klbl">${{l}}</div></div>`
  ).join('');
}}

function calcNationalLeg() {{
  const leg={{}};
  Object.values(STATS_UF).forEach(s=>{{
    Object.entries(s.leg||{{}}).forEach(([k,v])=> leg[k]=(leg[k]||0)+v);
  }});
  return leg;
}}

/* ── GRÁFICO DONUT ───────────────────────────────────────────────────────── */
function buildDonut(leg) {{
  const labels = Object.keys(leg);
  const vals   = Object.values(leg);
  const colors = labels.map(l => LEG_COLORS[l]||'#888');
  const ctx = document.getElementById('chart-donut').getContext('2d');
  if(cDonut) cDonut.destroy();
  cDonut = new Chart(ctx,{{
    type:'doughnut',
    data:{{labels,datasets:[{{data:vals,backgroundColor:colors,
              borderWidth:1,borderColor:'#0a1a0d'}}]}},
    options:{{
      responsive:true,maintainAspectRatio:false,animation:{{duration:350}},cutout:'52%',
      plugins:{{
        legend:{{position:'right',labels:{{color:'#86efac',font:{{size:9}},
                  boxWidth:11,padding:5}}}},
        tooltip:{{
          backgroundColor:'#030a05',borderColor:'#163520',borderWidth:1,
          titleColor:'#bbf7d0',bodyColor:'#86efac',
          callbacks:{{label:c=>`${{c.label}}: ${{c.raw.toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha`}}
        }}
      }}
    }}
  }});
}}

/* ── RANKING ─────────────────────────────────────────────────────────────── */
function setRankMode(m) {{
  rankMode=m;
  document.getElementById('rbtn-area').classList.toggle('on',m==='area');
  document.getElementById('rbtn-pct').classList.toggle('on',m==='pct');
  document.getElementById('rank-lbl').textContent = m==='area' ? 'Área total (ha)' : '% do município';
  buildRanking();
  buildUFBar();
}}

function buildRanking() {{
  const data = rankMode==='area' ? RANK_AREA : RANK_PCT;
  const top  = data.slice(0,50);
  const maxV = top.length ? (rankMode==='area'?top[0].area_ha:top[0].pct_mun_coberta) : 1;
  document.getElementById('ranking-mun').innerHTML = top.map((r,i) => {{
    const val  = rankMode==='area'
      ? r.area_ha.toLocaleString('pt-BR',{{maximumFractionDigits:0}})+' ha'
      : r.pct_mun_coberta.toFixed(1)+'%';
    const raw  = rankMode==='area' ? r.area_ha : r.pct_mun_coberta;
    const pct  = (raw/maxV*100).toFixed(0);
    return `<div class="ritem">
      <span class="rnum">${{i+1}}</span>
      <div style="flex:1">
        <div style="display:flex;justify-content:space-between">
          <div class="rname">${{r.municipio||r.cd_mun}}<small>${{r.uf}}</small></div>
          <span class="rval">${{val}}</span>
        </div>
        <div class="rbar"><div class="rfill" style="width:${{pct}}%"></div></div>
      </div>
    </div>`;
  }}).join('');
}}

function buildUFBar() {{
  const data = Object.entries(STATS_UF)
    .sort((a,b) => rankMode==='area'
      ? b[1].area - a[1].area
      : b[1].pct_uf - a[1].pct_uf)
    .slice(0,20);
  const labels = data.map(d=>d[0]);
  const vals   = data.map(d => rankMode==='area' ? d[1].area : d[1].pct_uf);
  const ctx = document.getElementById('chart-uf-bar').getContext('2d');
  if(cUFBar) cUFBar.destroy();
  cUFBar = new Chart(ctx,{{
    type:'bar',
    data:{{labels,datasets:[{{label:'',data:vals,
      backgroundColor:'rgba(74,222,128,.55)',borderRadius:3,borderSkipped:false}}]}},
    options:{{
      indexAxis:'y',responsive:true,maintainAspectRatio:false,animation:{{duration:350}},
      plugins:{{legend:{{display:false}},
        tooltip:{{backgroundColor:'#030a05',borderColor:'#163520',borderWidth:1,
          titleColor:'#bbf7d0',bodyColor:'#86efac',
          callbacks:{{label:c=>rankMode==='area'
            ? c.raw.toLocaleString('pt-BR',{{maximumFractionDigits:0}})+' ha'
            : c.raw.toFixed(1)+'%'}}}}
      }},
      scales:{{
        x:{{ticks:{{color:'#3d6b4a',font:{{size:8}}}},grid:{{color:'#0f2214'}}}},
        y:{{ticks:{{color:'#86efac',font:{{size:9}}}},grid:{{color:'#0f2214'}}}}
      }}
    }}
  }});
}}

/* ── SÉRIE HISTÓRICA ─────────────────────────────────────────────────────── */
function buildSeries() {{
  const anos  = SERIE.map(d=>d.ano);
  const acum  = SERIE.map(d=>d.area_acum_ha/1e6);
  const anual = SERIE.map(d=>d.area_ha/1e6);

  const co = ctx => ({{
    responsive:true,maintainAspectRatio:false,animation:{{duration:350}},
    plugins:{{legend:{{display:false}},
      tooltip:{{backgroundColor:'#030a05',borderColor:'#163520',borderWidth:1,
        titleColor:'#bbf7d0',bodyColor:'#86efac',
        callbacks:{{label:c=>`${{c.raw.toFixed(2)}} M ha`}}}}
    }},
    scales:{{
      x:{{ticks:{{color:'#3d6b4a',font:{{size:8}},maxRotation:45}},grid:{{color:'#0f2214'}}}},
      y:{{ticks:{{color:'#3d6b4a',font:{{size:9}}}},grid:{{color:'#0f2214'}}}}
    }}
  }});

  const ctx1 = document.getElementById('chart-serie-acum').getContext('2d');
  if(cSerieAcum) cSerieAcum.destroy();
  cSerieAcum = new Chart(ctx1,{{
    type:'line',
    data:{{labels:anos,datasets:[{{label:'Acumulado',data:acum,
      borderColor:'#4ade80',backgroundColor:'rgba(74,222,128,.12)',
      fill:true,tension:0.35,pointRadius:0,borderWidth:2}}]}},
    options:co()
  }});

  const ctx2 = document.getElementById('chart-serie-anual').getContext('2d');
  if(cSerieAnual) cSerieAnual.destroy();
  cSerieAnual = new Chart(ctx2,{{
    type:'bar',
    data:{{labels:anos,datasets:[{{label:'Criações anuais',data:anual,
      backgroundColor:'rgba(96,165,250,.6)',borderRadius:3,borderSkipped:false}}]}},
    options:co()
  }});
}}

/* ── BIOMAS ──────────────────────────────────────────────────────────────── */
function buildBiomas() {{
  const biomas = Object.keys(BIOMA_DATA);
  const totals = biomas.map(b => Object.values(BIOMA_DATA[b]).reduce((a,c)=>a+c,0));
  const biColors = ['#4ade80','#f59e0b','#f87171','#60a5fa','#a78bfa','#2dd4bf'];

  const ctx = document.getElementById('chart-bioma').getContext('2d');
  if(cBioma) cBioma.destroy();
  cBioma = new Chart(ctx,{{
    type:'bar',
    data:{{labels:biomas,datasets:[{{
      data:totals.map(v=>v/1e6),
      backgroundColor:biColors,borderRadius:3,borderSkipped:false
    }}]}},
    options:{{
      responsive:true,maintainAspectRatio:false,animation:{{duration:350}},
      plugins:{{legend:{{display:false}},
        tooltip:{{backgroundColor:'#030a05',borderColor:'#163520',borderWidth:1,
          titleColor:'#bbf7d0',bodyColor:'#86efac',
          callbacks:{{label:c=>`${{c.raw.toFixed(2)}} M ha`}}}}
      }},
      scales:{{
        x:{{ticks:{{color:'#86efac',font:{{size:8}}}},grid:{{color:'#0f2214'}}}},
        y:{{ticks:{{color:'#3d6b4a',font:{{size:9}}}},grid:{{color:'#0f2214'}}}}
      }}
    }}
  }});

  // Breakdown por bioma
  const cats = [...new Set(Object.values(BIOMA_DATA).flatMap(d=>Object.keys(d)))];
  document.getElementById('bioma-detail').innerHTML = biomas.map((b,bi)=>{{
    const rows = cats.map(c=>{{
      const v = BIOMA_DATA[b][c]||0;
      if(!v) return '';
      return `<div class="srow">
        <span class="slbl" style="display:flex;align-items:center;gap:5px">
          <span style="width:7px;height:7px;border-radius:50%;display:inline-block;
                        background:${{LEG_COLORS[c]||'#888'}}"></span>${{c}}</span>
        <span class="sval">${{v.toLocaleString('pt-BR',{{maximumFractionDigits:0}})}} ha</span>
      </div>`;
    }}).join('');
    return `<div style="margin-bottom:10px">
      <div style="font-size:11px;font-weight:600;color:${{biColors[bi]}};margin-bottom:5px">
        ${{b}} — ${{(totals[bi]/1e6).toFixed(2)}} M ha</div>
      ${{rows}}</div>`;
  }}).join('');
}}

/* ── TABS ────────────────────────────────────────────────────────────────── */
function switchTab(t) {{
  ['visao','ranking','historia','bioma'].forEach((id,i) => {{
    document.querySelectorAll('.tab')[i].classList.toggle('active',id===t);
    document.getElementById(`tc-${{id}}`).classList.toggle('active',id===t);
  }});
  if(t==='ranking') {{ buildRanking(); buildUFBar(); }}
  if(t==='historia') buildSeries();
  if(t==='bioma')    buildBiomas();
}}

/* ── INIT ────────────────────────────────────────────────────────────────── */
window.addEventListener('load', () => {{
  Chart.register(window['chartjs-plugin-annotation']||{{}});
  initMap();
  loadLayers();
}});
</script>
</body>
</html>"""

path_html = BASE_OUT / 'index.html'
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(HTML)

sz = path_html.stat().st_size / 1024
ok(f'WebMap gerado: {path_html.name} ({sz:.1f} KB)')
print(f'\n  Para GitHub Pages: publique os arquivos da pasta outputs/ na raiz do repo.')
print(f'  Estrutura recomendada do repositório:')
print(f'    ├── index.html')
print(f'    ├── geojson_uf.json')
print(f'    ├── geojson_municipios.json')
print(f'    └── geojson_cnfp.json')
print(f'\n  ✅ Tudo pronto! Ative GitHub Pages em Settings → Pages → Deploy from branch.')
